In [2]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC

from bs4 import BeautifulSoup
import re

sns.set_style('darkgrid')

In [3]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 781.4 KB


In [5]:
df.sample(5)

,review,sentiment
43985,FORBIDDEN PLANET is the best SF film from the ...,positive
34427,This was a great movie with a good story. My c...,positive
14652,"Partially from the perceived need, one feels, ...",positive
9371,"If you hit your teens in the 70s, as I did, yo...",negative
30447,In this tense and character-driven romantic tr...,positive


Nuestro conjunto de datos tiene dos columnas:

Review:  Reseña que escribió alguna persona acerca de una película (no sabemos a qué película fue, pero no es relevante para este análisis).

Sentiment: Si la reseña fue positiva o negativa (negative o positive)

Finalmente, veamos cuántas reseñas positivas y cuántas negativas tenemos:

In [6]:
df.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

### 25,000 reseñas positivas y 25,000 negativas

Reflexión inicial

Estamos por comenzar a hacer análisis de sentimiento usando Python. Nuestro modelo será capaz de determinar si una reseña es positiva o negativa. Por ejemplo, después de entrenar nuestro modelo, si le pedimos que analice el texto “I really loved this movie,” nuestro modelo debería indicarnos que se trata de una reseña positiva.

Esto podría dar la impresión de que la computadora está comprendiendo el texto o el lenguaje de la misma manera que lo haría un ser humano. Sin embargo, lo que haremos es algo que, aunque suene sorprendente, es fundamental en el procesamiento de lenguaje natural. Vamos a convertir los enunciados en una representación numérica. A esto se le conoce como vectorización del texto.

Mediante técnicas como TF-IDF, podemos transformar las palabras y frases en vectores, que son simplemente listas de números que representan la importancia de cada término en el contexto del texto. Estos vectores son los que el modelo utiliza para "comprender" el contenido y realizar predicciones sobre el sentimiento de la reseña.

Este proceso no implica comprensión del lenguaje en el sentido humano, sino una interpretación matemática que permite al modelo hacer inferencias basadas en patrones observados durante su entrenamiento.

Preparación de datos

Este proceso de entrenamiento será mucho más pesado que otros que hemos visto anteriormente. Por lo tanto, para fines demostrativos, reduciremos el tamaño de nuestros datos. Tomaremos 5,000 reseñas positivas y 5,000 negativas

In [7]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

Ahora procedemos a hacer nuestro split de datos de entrenamiento y de pruebas;

In [8]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

Si inspeccionamos train_y, vemos

In [9]:
train_y.value_counts()

sentiment
negative    3378
positive    3322
Name: count, dtype: int64

De texto a números

Ahora procederemos a transformar nuestros valores de texto a valores numéricos. Para esto, usaremos Scikit Learn. Específicamente, utilizaremos la clase TfidfVectorizer para transformar texto en una representación numérica utilizando el esquema TF-IDF (Term Frequency-Inverse Document Frequency).

Nuestro código es el siguiente:

In [10]:
tfidf = TfidfVectorizer(stop_words='english')
train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

TfidfVectorizer es una clase utilizada para convertir una colección de documentos de texto en una matriz de características.

TF-IDF es una técnica que combina dos estadísticas:

Term Frequency (TF): Frecuencia de un término en un documento, que mide la importancia del término en el documento en cuestión.

Inverse Document Frequency (IDF): Mide la importancia del término en lque a colección de documentos. Penaliza los términos que son muy comunes en todos los documentos.

Llama la atención el argumento “stop_words”. Este arugmento se utilizara para indicar que se deben eliminar las palabras comunes en inglés (como "the", "and", "is", etc.) que no aportan información relevante para el análisis.

Inspeccionemos un poco nuestros objetos antes y después de transformar:

In [11]:
train_x.shape

(6700,)

Nada que nos sorprenda aquí. En train_x tenemos 6,700 reseñas en formato de texto. Ahora, ya tenemos una nueva variable train_x_vector, el cual resultó de ejectutar fit_transform sobre train_x.

In [12]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

El tipo de dato scipy.sparse._csr.csr_matrix es una representación eficiente de una matriz dispersa en formato CSR (Compressed Sparse Row), que se utiliza para almacenar y manipular matrices grandes y dispersas.

¿Qué es una matriz dispersa?

Una matriz dispersa es una matriz en la que la mayoría de los elementos son cero. Almacenar todos estos ceros de forma explícita sería ineficiente tanto en términos de espacio como de tiempo. Por eso, en lugar de almacenar la matriz completa, se utiliza una representación que solo guarda los elementos no nulos y su posición, lo que reduce significativamente el uso de memoria y mejora la eficiencia de las operaciones.

Pandas tiene una forma de trabajar con matrices dispersas de tipo csr_matrix:

In [13]:
pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out())

,00,000,007,00am,00s,01,01pm,02,04,05,...,émigré,émigrés,était,étc,être,ísnt,île,önsjön,über,überwoman
6746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8499,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1685,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### No sabemos ni qué buscar…

Intentemos lo siguiente:

In [14]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

Esto nos muestra la primera reseña:

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

Ahora, busquemos este registro en nuestra matriz dispersa:

In [15]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]
primera_resenia

00           NaN
000          NaN
007          NaN
00am         NaN
00s          NaN
            ... 
ísnt         NaN
île          NaN
önsjön       NaN
über         NaN
überwoman    NaN
Name: 6746, Length: 44107, dtype: Sparse[float64, nan]

Convertir NaN a ceros

In [20]:
primera_resenia = primera_resenia.fillna(0)

Seguimos sin ver nada relevante. Tomemos esta serie y filtremos todo lo que no es cero:


In [21]:
primera_resenia[primera_resenia != 0]

13               0.45849
17               0.12824
actual          0.091601
actually        0.061461
age             0.088765
allowing         0.12824
beware          0.143046
br              0.124945
bunch           0.093128
deplorable      0.168137
disgusting      0.237945
entertaining    0.081088
expectations    0.103149
great           0.048868
happened        0.176853
hopes           0.114028
humor           0.084465
humorous        0.116516
including       0.087603
joked           0.168137
just            0.037675
know            0.053984
let              0.07195
like            0.036454
mature          0.126021
movie           0.276309
movies            0.0522
old             0.123513
pedophilia      0.172712
pg              0.260154
rated           0.206643
recommend       0.076198
references      0.226901
rent            0.093234
revolting       0.151971
safe            0.119732
sexual          0.098677
shouldn         0.108203
sister          0.095008
small           0.078916


¡Por fin vemos algo que nos on puros ceros!

Pero…

¿Qué significan todos estos números?

Estos números representan la importancia de cada palabra en la reseña que estamos analizando, basados en la técnica TF-IDF, la cual nos dice cuán relevante es una palabra dentro de una reseña en comparación con todas las demás reseñas. 

Por ejemplo, en la lista que obtuvimos, la palabra "movie" tiene un valor de 0.276309, lo que indica que es una palabra bastante importante en esta reseña específica. Mientras más alto sea el número, más relevante es la palabra en ese contexto particular. Estos valores no significan que la computadora "entienda" las palabras, sino que usa estos números para hacer comparaciones y predicciones, como decidir si la reseña es positiva o negativa.

Quizá estés pensando

¿De dónde salieron estos números?

Estos números se obtuvieron mediante un proceso llamado TF-IDF (Term Frequency-Inverse Document Frequency). Este proceso tiene dos partes:

Frecuencia de Término (TF): Primero, se cuenta cuántas veces aparece cada palabra en la reseña. Si una palabra aparece muchas veces en una reseña, se considera importante en esa reseña específica.

Frecuencia Inversa de Documento (IDF): Luego, se verifica cuántas veces aparece esa misma palabra en todas las reseñas del conjunto de datos. Si una palabra aparece en muchas reseñas, se considera menos importante porque es común y no ayuda mucho a distinguir una reseña de otra.

El valor final de TF-IDF es una combinación de estos dos factores. Las palabras que son frecuentes en una reseña pero raras en el resto del conjunto de datos obtienen un puntaje alto, lo que indica que son particularmente relevantes para esa reseña. Por otro lado, las palabras comunes en todas las reseñas obtienen un puntaje bajo. Estos puntajes se utilizan luego para que el modelo pueda hacer predicciones sobre el sentimiento de la reseña.

Entrenamiento

Usaremos un modelo llamado Máquinas de Soporte Vectorial o Máquinas de Vectores de Soporte.

SVM es adecuado para el análisis de sentimiento porque maneja bien la alta dimensionalidad del texto, es efectivo en clasificaciones binarias como positivo o negativo, y generaliza bien a nuevos datos, asegurando predicciones precisas incluso con reseñas no vistas.

In [22]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


### ¡Listo! Ahora probemos este modelo entrenado con algunas reseñas nuevas

In [23]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all'])))

['positive']
['positive']
['negative']


### ¿Nada mal no?

Evaluación

Debemos tener métricas objetivas de qué tan bien está nuestro modelo. Veamos algunas:

In [24]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

Precisión

La más simple de todas:

In [25]:
print(svc.score(test_x_vector, test_y))

0.8706060606060606


El comando svc.score(test_x_vector, test_y) calcula la precisión del modelo SVM que acabamos de entrenar.

La precisión es la proporción de predicciones correctas entre todas las predicciones realizadas.

En este caso, el output 0.8706060606060606 significa que el modelo acertó aproximadamente el 87.06% de las veces al predecir si una reseña era positiva o negativa en el conjunto de prueba (test_x_vector). Es una métrica simple pero útil para tener una idea general de qué tan bien está funcionando el modelo.

F1 Score

In [26]:
f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87400413, 0.86701962])

Aquí calculamos la métrica F1 para cada una de las clases especificadas: 'positive' y 'negative'. La métrica F1 es la media armónica de la precisión y el recall, lo que la convierte en una métrica más equilibrada para evaluar el rendimiento del modelo, especialmente en casos de clases desbalanceadas.

El output array([0.87400413, 0.86701962]) nos da el puntaje F1 para ambas clases:

0.87400413 es el F1 score para la clase 'positive'. 0.86701962 es el F1 score para la clase 'negative'. Estos valores indican que el modelo tiene un rendimiento similar para ambas clases, con un ligero mejor rendimiento en la predicción de reseñas positivas (0.874) en comparación con las negativas (0.867).

El recall (también conocido como sensibilidad o tasa de verdaderos positivos) es una métrica de evaluación que mide la capacidad de un modelo para identificar correctamente todos los casos positivos entre los que realmente son positivos.

En términos simples, el recall responde a la pregunta: "De todos los casos que realmente son positivos, ¿cuántos fueron correctamente identificados por el modelo?"

Un recall alto significa que el modelo está capturando la mayoría de los verdaderos positivos, pero no necesariamente indica que las predicciones negativas sean precisas. Es una métrica crucial cuando la prioridad es no perder casos positivos, como en la detección de enfermedades o fraudes.

Reporte de clasificación

In [27]:
print(classification_report(test_y, svc.predict(test_x_vector), labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.87      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



El comando classification_report genera un informe que muestra varias métricas clave para evaluar el rendimiento del modelo en la clasificación de reseñas como positivas o negativas. En este caso, el modelo tiene una precisión de 0.87 para la clase 'positive' y de 0.88 para la clase 'negative', lo que indica que es ligeramente más preciso al identificar reseñas negativas. El recall para 'positive' es 0.88, mientras que para 'negative' es 0.86, sugiriendo que el modelo es algo mejor capturando reseñas positivas. El F1-Score, que equilibra precisión y recall, es de 0.87 para ambas clases, reflejando un rendimiento uniforme. El soporte, que indica cuántas reseñas de cada tipo estaban en el conjunto de prueba, es de 1678 para 'positive' y 1622 para 'negative'. La precisión global del modelo es del 87%, lo que significa que acertó en el 87% de las predicciones en el conjunto de prueba. El promedio macro, que toma el promedio de las métricas sin considerar el tamaño de las clases, y el promedio ponderado, que sí considera el tamaño de las clases, ambos dan un valor de 0.87, confirmando que el modelo tiene un rendimiento equilibrado y consistente entre ambas clases.

Matriz de confusión

In [28]:
conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1481,  197],
       [ 230, 1392]])

Una matriz de confusión es una herramienta para evaluar el rendimiento del modelo clasificando datos en categorías. En este caso, la matriz de confusión se ha generado para las clases 'positive' y 'negative', y el output es un array 2x2 que se interpreta de la siguiente manera:

La primera fila corresponde a las predicciones de la clase 'positive'.

El valo 1481 indica el número de reseñas que fueron correctamente clasificadas como 'positive' (Verdaderos Positivos).

El valor 197 indica el número de reseñas que fueron incorrectamente clasificadas como 'negative' cuando en realidad eran 'positive' (Falsos Negativos).

La segunda fila corresponde a las predicciones de la clase 'negative'.

El valor 230  indica el número de reseñas que fueron incorrectamente clasificadas como 'positive' cuando en realidad eran 'negative' (Falsos Positivos).

El valor 1392  indica el número de reseñas que fueron correctamente clasificadas como 'negative' (Verdaderos Negativos).

Siguientes pasos

Ahora te toca a ti. Implementa otros modelos para verificar si máquinas de vectores de soporte fue la mejor elección. Puedes probar con estos modelos:

LogisticRegression

GaussianNB

DecisionTreeClassifier